# Example of how to simulate light curves with the F-CAMs

In this example we will use a realistic star catalogue of stars from the PLATO Input Catalogue (PIC) version 1.1.0. Since the F-CAMs both has a cycle time of 2.5 second (2.1 exposure and 0.4 readout) we will only consider the brighest targets in the PIC, namely the P2 sample having $V<8.5$ mag, and we have selected only stars from the South PLATO Field (SPF) and having a N-CAM visibility of all 24 cameras. Hence, stars that all lies within the FOV of the F-CAMs as these share the pointing with the spacecraft platform/payload. In this example we use some generic function to be found in the simulation class "simfile".

In [1]:
#!/usr/bin/env Python3

import os
import warnings
import numpy as np
import matplotlib.pyplot as plt
from platosim.simulation import Simulation
from platosim.plot import *
import platosim.referenceFrames as rf

warnings.simplefilter("ignore")

In [2]:
# CONFIGURE SIMULATION

# User defined variables
simName      = 'sim0'
dataDir      = os.getcwd()
relativePath = '../platonium/examples/fastCamPhotometry'   # Relative path from PlatoSim to input files

# Start setting up the simulation
inputfile = dataDir + '/inputfile.yaml'
sim = Simulation(simName, configurationFile=inputfile, outputDir=dataDir)

# Set number of exposures
sim["ObservingParameters/NumExposures"] = 15000
sim["ObservingParameters/CycleTime"] = 2.5

# Set subfield size (min 8 pixels to perform photometry)
numRowsCols = sim["SubField/NumRows"] = sim["SubField/NumColumns"] = 8

# Control what to save as output
sim["ControlHDF5Content/WriteSubPixelImages"] = "yes"

# Set the group ID to F-CAMS
sim["Telescope/GroupID"] = "Fast"

# Set control parameters
sim["ControlHDF5Content/WritePixelMaps"]      = "yes"
sim["ControlHDF5Content/WriteStarPositions"]  = "yes"
sim["ControlHDF5Content/WriteBiasMaps"]       = "no"
sim["ControlHDF5Content/WriteSmearingMaps"]   = "no"
sim["ControlHDF5Content/WriteThroughputMaps"] = "no"
sim["ControlHDF5Content/WriteFlatfieldMap"]   = "no"
sim["ControlHDF5Content/WriteSubPixelImages"] = "no"

In [3]:
# LOAD AND CREATE USER DEFINED STAR CATALOGUE

# Loading PIC targets
pic_tar = np.loadtxt(dataDir + '/starcat-SPF-P2-Targets.txt')
ID  = pic_tar[:,0].astype(int)
ra  = pic_tar[:,1]
dec = pic_tar[:,2]
mag = pic_tar[:,3]

# Loading PIC contaminants
pic_con = np.loadtxt(dataDir + '/starcat-SPF-P2-Contaminants.txt')
ID_con   = pic_con[:,0].astype(int)
ra_con   = pic_con[:,1]
dec_con  = pic_con[:,2]
mag_con  = pic_con[:,3]

OSError: /lhome/nicholas/software/platonium/examples/Photometry/starcat-SPF-P2-Targets.txt not found.

In [ ]:
# MAKE CHECKS BEFORE RUNNING SIMULATION

raPlatform  = np.deg2rad(sim["ObservingParameters/RApointing"])
decPlatform = np.deg2rad(sim["ObservingParameters/DecPointing"])

pixelSize       = float(sim["CCD/PixelSize"])
focalLength     = float(sim["Camera/FocalLength/ConstantValue"]) * 1000.0  # [m] -> [mm]
focalPlaneAngle = np.deg2rad(float(sim["Camera/FocalPlaneOrientation/ConstantValue"]))
solarPanelAngle = np.deg2rad(float(sim["Platform/SolarPanelOrientation"]))

azimuth = np.deg2rad(sim["Telescope/AzimuthAngle"])
tilt    = np.deg2rad(sim["Telescope/TiltAngle"])

# First let's have a look at the CCDs
drawCCDsInFocalPlane(pixelSize, normal=False)

# Plot star on the F-CAMs CCDs

fig, ax = plt.subplots(figsize=(15, 9))
drawCCDsInSkyMollweide(fig, raPlatform, decPlatform, solarPanelAngle, tilt, azimuth, 
                       focalPlaneAngle, focalLength, pixelSize, normal=False)
drawStarsInSkyMollweide(fig, ra, dec)

In [ ]:
# We see that not all stars from out catalogue falls on a CCD
# We can use the following function to limit our sample to stars that do fall on a CCD

subfieldIsOnCCD = np.zeros_like(ra, dtype=bool)

for i in range(len(ra)):
    subfieldIsOnCCD[i] = sim.setSubfieldAroundCoordinates(np.deg2rad(ra[i]), np.deg2rad(dec[i]),
                                                          numRowsCols, numRowsCols, normal=False)
    
# Now select targets that do fall on a CCD
ID  = ID[subfieldIsOnCCD].astype(int)
ra  = ra[subfieldIsOnCCD]
dec = dec[subfieldIsOnCCD]
mag = mag[subfieldIsOnCCD]

# Plot all stars that is on the F-CAMs CCDs

fig, ax = plt.subplots(figsize=(15, 9))
drawCCDsInSkyMollweide(fig, raPlatform, decPlatform, solarPanelAngle, tilt, azimuth, 
                       focalPlaneAngle, focalLength, pixelSize, normal=False)
drawStarsInSkyMollweide(fig, ra, dec)

In [ ]:
# INCLUDE STAR CATALOGUE

# As an example we now pick the first star in the catalogue
targetNo = 2

# Create arrays
idTarget  = np.array(ID[targetNo])
raTarget  = np.array(ra[targetNo])
decTarget = np.array(dec[targetNo])
magTarget = np.array(mag[targetNo])

# Find corresponding cantaminants
dex = np.where(ID_con == ID[targetNo])[0]
idCon  = np.take(ID_con,  dex)
magCon = np.take(mag_con, dex)
raCon  = np.take(ra_con,  dex)
decCon = np.take(dec_con, dex)
numCon = len(dex)

# Make a star catalog with all the stars in this subfield
raStarCatalog  = np.append(raTarget,  raCon)
decStarCatalog = np.append(decTarget, decCon)
magStarCatalog = np.append(magTarget, magCon)
idStarCatalog  = np.append(idTarget,  idCon)

# Save catalog and load it into the inputfile
starCatalog = np.transpose([raStarCatalog, decStarCatalog, magStarCatalog, np.arange(1, len(idStarCatalog)+1)])
starCatalogFile = dataDir + '/starCatalog.txt'
sim["ObservingParameters/StarCatalogFile"] = starCatalogFile
np.savetxt(starCatalogFile, starCatalog, fmt=['%11.6f', '%11.6f', '%8.4f', '%i'])

# Lastly we set the subfield around the target's coordinates

subfieldIsOnCCD = sim.setSubfieldAroundCoordinates(np.deg2rad(raTarget), np.deg2rad(decTarget),
                                                   numRowsCols, numRowsCols, normal=False)

print(subfieldIsOnCCD)

In [ ]:
# INCLUDE PHOTOMETRY

# Create and include photometry list of targets
photometryFile = dataDir + '/photometryFile.txt'
np.savetxt(photometryFile, np.array([1]), fmt='%i')
sim['Photometry/IncludePhotometry'] = True
sim["Photometry/TargetFileName"] = photometryFile

In [ ]:
# INCLUDE VARIABILITY

# First we create the varsourceFile
varsourceFile = dataDir + '/varsourceFile.txt'
np.savetxt(varsourceFile, np.array(['1 ' + relativePath + '/varsource.txt']), fmt=['%s'])
# Include a hot-Jupiter transiting a Sun-like star as variable source
sim['Sky/IncludeVariableSources'] = True
sim["Sky/VariableSourceList"] = varsourceFile

In [ ]:
# SELECT SUBFIELD AND RUN SIMULATION

xFP, yFP = rf.skyToFocalPlaneCoordinates(np.deg2rad(raTarget), np.deg2rad(decTarget), raPlatform, decPlatform,
                                      solarPanelAngle, tilt, azimuth,
                                      focalPlaneAngle, focalLength)

distanceOA = np.rad2deg(rf.gnomonicRadialDistanceFromOpticalAxis(xFP, yFP, focalLength))

print(distanceOA)

In [ ]:
# RUN SIMULATION

simFile = sim.run(removeOutputFile=True)

# Remove produced log and yaml file
output_sim = dataDir + '/' + simName
os.remove(output_sim + '.log')
os.remove(output_sim + '.yaml')
#os.remove(output_sim + '.hdf5')

In [ ]:
# Extract the CCD image and the star positions
im = simFile.getImage(0)

# Fetch light curve
lc = simFile.getLightCurve(1)

#Plot imagette and pixel mask
# Show image (cmap suggestions: "hot", "gnuplot2","magma", and "Spectral_r")
title = 'Imagette of PIC {0} ({1:.2f} mag)'.format(idTarget, magTarget)
simFile.showImage(0, showStarPositions='PIC', clipPercentile=8, showMaskOfStarID=1,
                              useTitle=title, colorMap='Spectral_r', showGrid=True) 


# Plot light curve with build-in function
plt.figure()
plt.plot(lc[0], lc[1], 'k-')
plt.xlabel('Time [s]')
plt.ylabel('FLux [ADU]')
plt.show()